

*   Batch gradient descent
*   tanhx and softmax usage
*   dropout mask
*   no error calculation because cross entropy calculation is hard :')






In [ ]:
import numpy as np,sys
# np.random.seed(1)

In [ ]:
from keras.datasets import mnist
(x_train,y_train),(x_test,y_test)=mnist.load_data()


In [ ]:
print(x_train.min(),x_train.max()) ##not corrupted

0 255


In [ ]:
print(x_train.shape,y_train.shape)

(60000, 28, 28) (60000,)


In [ ]:
images,labels=x_train[:1000].reshape(1000,28*28)/255,y_train[:1000]

In [ ]:
one_hot_labels=np.zeros((len(labels),10))
print(labels[:10])

for i,l in enumerate(labels):
  one_hot_labels[i][l]=1

labels=one_hot_labels

[5 0 4 1 9 2 1 3 1 4]


In [ ]:
test_images,test_labels=x_test.reshape(len(x_test),28*28)/255,y_test

In [ ]:
one_hot_test_labels=np.zeros((len(test_labels),10))
for i,l in enumerate(test_labels):
  one_hot_test_labels[i][l]=1

test_labels=one_hot_test_labels
# print(test_labels[:5])

In [ ]:
##Activation functions

def tanh(x):
  return np.tanh(x)

def tanhderivative(x):
  return 1-(x)**2

def softmax(x):
  # Subtracting the maximum value for numerical stability
  exp_scores = np.exp(x)
  return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

In [ ]:
alpha,iterations,hidden_size= (2,300,100)
pixels_per_image,num_lables=(784,10)
batch_size=100

In [ ]:
weight_0_1=0.02*np.random.random((pixels_per_image,hidden_size))-0.01
weight_1_2=0.2*np.random.random((hidden_size,num_lables))-0.1


(100, 784) (100, 100) (100, 1) (784, 100) (100, 10)



In [ ]:
for j in range(iterations):
  correct_cnt=0
  for i in range(len(images)//batch_size):
      batch_start,batch_end=(i*batch_size),((i+1)*batch_size)
      ip=images[batch_start:batch_end]
      op=labels[batch_start:batch_end]
      layer_0=ip

      layer_1=tanh(np.dot(layer_0,weight_0_1))

      dropout_mask=np.random.randint(2,size=layer_1.shape)
      layer_1*=dropout_mask*2
      layer_2=softmax(np.dot(layer_1,weight_1_2))

      for k in range(batch_size):
              correct_cnt += int(np.argmax(layer_2[k:k+1]) == \
              np.argmax(labels[batch_start+k:batch_start+k+1]))

      # Correct delta_layer_2 for softmax with cross-entropy
      delta_layer_2=(op-layer_2)/(batch_size*100)
      delta_layer_1=delta_layer_2.dot(weight_1_2.T)*tanhderivative(layer_1)
      delta_layer_1*=dropout_mask

      weight_1_2+=alpha*layer_1.T.dot(delta_layer_2)
      weight_0_1+=alpha*layer_0.T.dot(delta_layer_1)

      # if i==0:
      #   print(layer_0.shape,layer_1.shape,layer_2.shape,weight_0_1.shape,weight_1_2.shape)
  # if j==0:
  #   print("mean abs of delta_layer_2: ", np.mean(np.abs(delta_layer_2)))
  #   print(delta_layer_2[:10])
  test_correct_cnt = 0
  for i in range(len(test_images)):
      layer_0 = test_images[i:i+1]
      layer_1 = tanh(np.dot(layer_0,weight_0_1))
      # No dropout on test set
      layer_2 = np.dot(layer_1,weight_1_2)
      test_correct_cnt += int(np.argmax(layer_2) == \
  np.argmax(test_labels[i:i+1]))
  if(j % 10 == 0):
    print("\n"+ "I:" + str(j) + \
      " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
      " Train-Acc:" + str(correct_cnt/float(len(images))))


I:0 Test-Acc:0.8132 Train-Acc:0.858

I:10 Test-Acc:0.822 Train-Acc:0.866

I:20 Test-Acc:0.8307 Train-Acc:0.878

I:30 Test-Acc:0.8364 Train-Acc:0.877

I:40 Test-Acc:0.8415 Train-Acc:0.88

I:50 Test-Acc:0.8444 Train-Acc:0.893

I:60 Test-Acc:0.8487 Train-Acc:0.9

I:70 Test-Acc:0.8511 Train-Acc:0.906

I:80 Test-Acc:0.8539 Train-Acc:0.908

I:90 Test-Acc:0.8556 Train-Acc:0.92

I:100 Test-Acc:0.8576 Train-Acc:0.909

I:110 Test-Acc:0.8597 Train-Acc:0.907

I:120 Test-Acc:0.8611 Train-Acc:0.92

I:130 Test-Acc:0.8628 Train-Acc:0.923

I:140 Test-Acc:0.8629 Train-Acc:0.925

I:150 Test-Acc:0.8642 Train-Acc:0.927

I:160 Test-Acc:0.8644 Train-Acc:0.935

I:170 Test-Acc:0.8658 Train-Acc:0.929

I:180 Test-Acc:0.867 Train-Acc:0.938

I:190 Test-Acc:0.8682 Train-Acc:0.941

I:200 Test-Acc:0.8687 Train-Acc:0.936

I:210 Test-Acc:0.8701 Train-Acc:0.944

I:220 Test-Acc:0.8706 Train-Acc:0.942

I:230 Test-Acc:0.8707 Train-Acc:0.94

I:240 Test-Acc:0.8723 Train-Acc:0.947

I:250 Test-Acc:0.8726 Train-Acc:0.945

I:26